# Pipeline обработки данных.

Чтение очищенного датасета. (датасет чистим в `eda.ipynb`)

## Задание 1 предобработка представлен в `eda.ipynb`

In [29]:
import pandas as pd

In [30]:
dataset_path = "dataset_clean.csv"

df = pd.read_csv(dataset_path, parse_dates=["TS"])

df.head()

,TS,Lab1_G1_N1,Lab1_G1_N2,Lab1_G1_N3,Lab1_G1_P2,Lab1_G1_T4ср,Lab1_G1_T1,Lab1_G1_T607,Lab1_G1_T600,Lab1_G1_T638,...,Lab1_AVOM_AVOMN1,Lab1_Kp,Lab1_hGPA,Lab1_Kran_5,Lab1_Kran_2,Lab1_Kran_6,Lab1_U_Kran_GPA_A_APK,Lab1_Kran_1,Lab1_Kran_4,Lab1_q
0,2022-12-02 18:59:59.999998+00:00,8726,11490,5000,15.35,662.7,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
1,2022-12-02 19:00:59.999998+00:00,8726,11489,4993,15.35,662.4,-16.4,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
2,2022-12-02 19:01:59.999999+00:00,8725,11493,5001,15.35,663.1,-16.0,36.1,67.0,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
3,2022-12-02 19:02:59.999999+00:00,8726,11493,5011,15.34,663.0,-16.1,36.1,67.1,79.8,...,1,0.92,0.27,0,1,0,0,1,0,50.8
4,2022-12-02 19:04:00+00:00,8728,11496,5003,15.33,663.4,-15.9,36.1,67.1,80.0,...,1,0.92,0.27,0,1,0,0,1,0,50.8


## Задание 2. Расчет показателей Хёрста и Ляпунова.

In [31]:
import numpy as np
import nolds

Подготовим вспомогательную функцию: ряд нормализуется. Константные ряды не анализируются, потому что для них R/S-анализ и показатель Ляпунова неинформативны.

In [32]:
def prepare_series(series):
    x = series.to_numpy()

    std = x.std()
    if std == 0:
        return None

    return (x - x.mean()) / std

Датасет является многоканальным временным рядом: в каждый момент времени измеряется несколько параметров установки. Поэтому на данном этапе каждый динамический числовой признак рассматривается как отдельный одномерный временной ряд.

Временная колонка `TS` используется только для упорядочивания наблюдений и не включается в расчёт показателей.

In [33]:
df_tda = df.sort_values("TS").reset_index(drop=True)

numeric_cols = df_tda.select_dtypes(include="number").columns.tolist()
numeric_cols = [
    col for col in numeric_cols
    if df_tda[col].nunique(dropna=False) > 1
]

print(f"Рядов для анализа: {len(numeric_cols)}")

Рядов для анализа: 84


Для каждого ряда считаем показатель Хёрста методом R/S-анализа и крупнейший показатель Ляпунова методом Розенштейна (`lyap_r`).

*пояснение*:

Для оценки показателя Ляпунова был использован метод Розенштейна, поскольку он позволяет оценить крупнейший показатель
Ляпунова непосредственно по одномерному временному ряду без задания аналитической модели системы. Метод основан на
реконструкции фазового пространства с помощью временных задержек и последующем анализе скорости расхождения близких
траекторий. Положительное значение крупнейшего показателя Ляпунова интерпретируется как признак чувствительности к
начальным условиям и возможной хаотической динамики, тогда как неположительное значение не подтверждает наличие
хаотического поведения. Такой подход является практичным для экспериментальных и промышленных временных рядов, где
доступны только наблюдения датчиков, но неизвестны уравнения порождающего процесса.

Для предварительной оценки показателя Ляпунова параметры lag и min_tsep подбирались встроенными эвристиками библиотеки nolds: задержка оценивается по автокорреляции, а минимальное временное разделение — по среднему периоду сигнала. На следующем этапе параметры вложения будут подбираться более подробно.

In [ ]:
results = []

# Каждый числовой признак рассматриваем как отдельный одномерный временной ряд.
for col in numeric_cols:
    x = prepare_series(df_tda[col])

    # Если ряд константный, показатели для него не считаются.
    if x is None:
        results.append({
            "column": col,
            "hurst_rs": np.nan,
            "lyapunov": np.nan,
            "error": "constant series",
        })
        continue

    try:
        hurst = nolds.hurst_rs(
            x,
            fit="poly",
            corrected=True,
            unbiased=True,
        )
    except Exception as exc:
        hurst = np.nan
        hurst_error = str(exc)
    else:
        hurst_error = ""

    try:
        lyapunov = nolds.lyap_r(
            x,
            lag=1,
            fit="poly",
        )
    except Exception as exc:
        lyapunov = np.nan
        lyapunov_error = str(exc)
    else:
        lyapunov_error = ""

    # Сохраняем численные значения и возможные ошибки расчета для диагностики.
    results.append({
        "column": col,
        "hurst_rs": hurst,
        "lyapunov": lyapunov,
        "error": "; ".join(
            err for err in [hurst_error, lyapunov_error]
            if err
        ),
    })

c:\Users\user\Desktop\Топология_курсовая\.venv\Lib\site-packages\nolds\measures.py:263: RuntimeWarning: signal has very low mean frequency, setting min_tsep = 359
  warnings.warn(msg.format(min_tsep), RuntimeWarning)


| Колонка    | Пояснение                                                                                                   |
| ---------- | ----------------------------------------------------------------------------------------------------------- |
| `column`   | Название исследуемого признака (временного ряда / сенсора / параметра установки)                            |
| `hurst_rs` | Показатель Херста, характеризующий наличие долгосрочной памяти и степень персистентности временного ряда    |
| `lyapunov` | Оценка наибольшего показателя Ляпунова, характеризующая чувствительность динамики ряда к начальным условиям |
| `error`    | Текст ошибки вычисления метрик (если расчёт для признака завершился неуспешно)                              |


In [35]:
process_report = pd.DataFrame(results)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
process_report

,column,hurst_rs,lyapunov,error
0,Lab1_G1_N1,0.896420,0.066899,
1,Lab1_G1_N2,0.892538,0.078063,
2,Lab1_G1_N3,0.485665,0.042443,
3,Lab1_G1_P2,0.744458,0.065301,
4,Lab1_G1_T4ср,0.885607,0.077858,
5,Lab1_G1_T1,0.902912,0.070435,
6,Lab1_G1_T607,0.723786,0.029808,
7,Lab1_G1_T600,0.906647,0.070207,
8,Lab1_G1_T638,0.937976,0.071134,
9,Lab1_G1_T606,0.883686,0.069799,


Добавим качественную интерпретацию.

| Термин | Семантически | Показатель Хёрста |
|---|---|---|
| **Статистически устойчивый / стационарный процесс** | По конспекту: статистические характеристики процесса устойчивы во времени | $$H < 0$$ или $$H > 1$$ |
| **Антиперсистентный процесс** | Ряд часто меняет направление: после роста вероятен спад, после спада — рост | $$0 \leq H \leq 0.4$$ |
| **Слабая антиперсистентность, возможна хаотическая динамика** | Возвратное поведение выражено слабо; возможны элементы сложной динамики | $$0.4 < H \leq 0.5$$ |
| **Слабая персистентность, возможна хаотическая динамика** | Долговременная память выражена слабо; возможны элементы сложной динамики | $$0.5 < H \leq 0.6$$ |
| **Персистентный процесс** | Ряд “помнит” прошлое: если рос, скорее продолжит расти; если снижался — продолжит снижаться | $$0.6 < H \leq 1$$ |


In [14]:
h = process_report["hurst_rs"]

process_report["hurst_type"] = np.select(
    [
        h.isna(),
        h < 0,
        h <= 0.4,
        (h > 0.4) & (h <= 0.5),
        (h > 0.5) & (h <= 0.6),
        (h > 0.6) & (h <= 1.0),
        h > 1.0,
    ],
    [
        "H < 1: статистически устойчивый / стационарный процесс",
        "значение H вне ожидаемого диапазона",
        "антиперсистентный процесс",
        "слабая антиперсистентность, возможна хаотическая динамика",
        "слабая персистентность, возможна хаотическая динамика",
        "персистентный процесс",
        "H > 1: статистически устойчивый / стационарный процесс",
    ],
    default="не определено",
)

process_report["lyapunov_type"] = np.select(
    [
        process_report["lyapunov"] > 0,
        process_report["lyapunov"] <= 0,
    ],
    [
        "признаки хаотической динамики",
        "хаотичность не подтверждается",
    ],
    default="не определено",
)

display(process_report.sort_values("lyapunov", ascending=False))

,column,hurst_rs,lyapunov,error,hurst_type,lyapunov_type
44,Lab1_G3_Pc1,0.441834,0.160560,,"слабая антиперсистентность, возможна хаотическ...",признаки хаотической динамики
45,Lab1_G3_Pc2,0.694835,0.123888,,персистентный процесс,признаки хаотической динамики
46,Lab1_G3_Pc3,0.556710,0.111137,,"слабая персистентность, возможна хаотическая д...",признаки хаотической динамики
55,Lab1_G3_ЗО_СТ,0.464293,0.110202,,"слабая антиперсистентность, возможна хаотическ...",признаки хаотической динамики
54,Lab1_G3_ПО_СТ,1.078973,0.103721,,H > 1: статистически устойчивый / стационарный...,признаки хаотической динамики
38,Lab1_G3_Lm,0.722304,0.101253,,персистентный процесс,признаки хаотической динамики
52,Lab1_G3_КВД,0.701866,0.100767,,персистентный процесс,признаки хаотической динамики
51,Lab1_G3_КНД,0.701866,0.100767,,персистентный процесс,признаки хаотической динамики
41,Lab1_G3_T638,0.903007,0.090695,,персистентный процесс,признаки хаотической динамики
47,Lab1_G3_T600,0.909729,0.089369,,персистентный процесс,признаки хаотической динамики


Сводные статистики нужны для общего вывода о природе порождающих процессов по всем временным рядам.

In [37]:
display(process_report[["hurst_rs", "lyapunov"]].describe())
display(process_report["hurst_type"].value_counts().to_frame("count"))
display(process_report["lyapunov_type"].value_counts().to_frame("count"))

,hurst_rs,lyapunov
count,84.000000,84.000000
mean,0.744078,0.057975
std,0.175706,0.025162
min,0.388588,0.018189
25%,0.606125,0.041532
50%,0.731952,0.049385
75%,0.874413,0.071383
max,1.362306,0.160560


,count
hurst_type,
персистентный процесс,60
"слабая персистентность, возможна хаотическая динамика",12
"слабая антиперсистентность, возможна хаотическая динамика",6
H > 1: статистически устойчивый / стационарный процесс,5
антиперсистентный процесс,1


,count
lyapunov_type,
признаки хаотической динамики,84


### Выводы по показателям Хёрста и Ляпунова



### Интерпретация результатов перед вложением в облако точек


## Проверка наличия линейной лаговой связи

In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr

try:
    from sklearn.metrics import normalized_mutual_info_score
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("sklearn не найден: взаимная информация считаться не будет")

sklearn не найден: взаимная информация считаться не будет


In [39]:
def discretize_by_quantiles(x, n_bins=10):
    """
    Дискретизация значений по квантилям.
    Нужна для расчета normalized mutual information.
    """
    x = pd.Series(x)

    # Если уникальных значений мало, уменьшаем число бинов
    unique_count = x.nunique()
    bins = min(n_bins, unique_count)

    if bins < 2:
        return None

    try:
        return pd.qcut(x, q=bins, labels=False, duplicates="drop")
    except ValueError:
        return None

In [40]:
def analyze_lag_linearity_for_series(x, column, max_lag=50, n_bins=10):
    """
    Анализ связи между x(t) и x(t + tau) для одного временного ряда.

    Pearson  — линейная зависимость.
    Spearman — монотонная зависимость.
    NMI      — общая зависимость, в том числе нелинейная.
    """
    rows = []

    max_lag = min(max_lag, len(x) // 3)

    for tau in range(1, max_lag + 1):
        x_t = x[:-tau]
        x_lag = x[tau:]

        # Если после сдвига данных слишком мало — пропускаем
        if len(x_t) < 30:
            continue

        # Pearson: линейная связь
        try:
            pearson_corr, pearson_p = pearsonr(x_t, x_lag)
        except Exception:
            pearson_corr, pearson_p = np.nan, np.nan

        # Spearman: монотонная связь
        try:
            spearman_corr, spearman_p = spearmanr(x_t, x_lag)
        except Exception:
            spearman_corr, spearman_p = np.nan, np.nan

        # Normalized Mutual Information: более общий тип зависимости
        if SKLEARN_AVAILABLE:
            x_t_disc = discretize_by_quantiles(x_t, n_bins=n_bins)
            x_lag_disc = discretize_by_quantiles(x_lag, n_bins=n_bins)

            if x_t_disc is not None and x_lag_disc is not None:
                nmi = normalized_mutual_info_score(x_t_disc, x_lag_disc)
            else:
                nmi = np.nan
        else:
            nmi = np.nan

        rows.append({
            "column": column,
            "tau": tau,
            "pearson_corr": pearson_corr,
            "pearson_abs": abs(pearson_corr) if not np.isnan(pearson_corr) else np.nan,
            "pearson_p": pearson_p,
            "spearman_corr": spearman_corr,
            "spearman_abs": abs(spearman_corr) if not np.isnan(spearman_corr) else np.nan,
            "spearman_p": spearman_p,
            "nmi": nmi,
        })

    return rows

In [41]:
lag_results = []

for col in numeric_cols:
    x = prepare_series(df_tda[col])

    if x is None:
        continue

    rows = analyze_lag_linearity_for_series(
        x=x,
        column=col,
        max_lag=50,
        n_bins=10,
    )

    lag_results.extend(rows)

lag_linearity_report = pd.DataFrame(lag_results)

display(lag_linearity_report.head())
print("Всего строк отчета:", len(lag_linearity_report))

,column,tau,pearson_corr,pearson_abs,pearson_p,spearman_corr,spearman_abs,spearman_p,nmi
0,Lab1_G1_N1,1,0.990183,0.990183,0.0,0.983904,0.983904,0.0,NaN
1,Lab1_G1_N1,2,0.981837,0.981837,0.0,0.972542,0.972542,0.0,NaN
2,Lab1_G1_N1,3,0.972523,0.972523,0.0,0.960188,0.960188,0.0,NaN
3,Lab1_G1_N1,4,0.964356,0.964356,0.0,0.950374,0.950374,0.0,NaN
4,Lab1_G1_N1,5,0.956212,0.956212,0.0,0.941499,0.941499,0.0,NaN


Всего строк отчета: 4200


In [42]:
def summarize_lag_linearity(group):
    """
    Сводка по одному временному ряду.
    """
    best_pearson_row = group.loc[group["pearson_abs"].idxmax()]
    best_spearman_row = group.loc[group["spearman_abs"].idxmax()]

    if group["nmi"].notna().any():
        best_nmi_row = group.loc[group["nmi"].idxmax()]
        max_nmi = best_nmi_row["nmi"]
        tau_max_nmi = best_nmi_row["tau"]
        pearson_at_max_nmi = best_nmi_row["pearson_corr"]
    else:
        max_nmi = np.nan
        tau_max_nmi = np.nan
        pearson_at_max_nmi = np.nan

    return pd.Series({
        "max_abs_pearson": best_pearson_row["pearson_abs"],
        "tau_max_pearson": best_pearson_row["tau"],
        "pearson_at_best_tau": best_pearson_row["pearson_corr"],

        "max_abs_spearman": best_spearman_row["spearman_abs"],
        "tau_max_spearman": best_spearman_row["tau"],
        "spearman_at_best_tau": best_spearman_row["spearman_corr"],

        "max_nmi": max_nmi,
        "tau_max_nmi": tau_max_nmi,
        "pearson_at_max_nmi": pearson_at_max_nmi,

        "mean_abs_pearson": group["pearson_abs"].mean(),
        "mean_abs_spearman": group["spearman_abs"].mean(),
        "mean_nmi": group["nmi"].mean(),
    })


linearity_summary = (
    lag_linearity_report
    .groupby("column")
    .apply(summarize_lag_linearity)
    .reset_index()
)

display(linearity_summary.head())

,column,max_abs_pearson,tau_max_pearson,pearson_at_best_tau,max_abs_spearman,tau_max_spearman,spearman_at_best_tau,max_nmi,tau_max_nmi,pearson_at_max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi
0,Lab1_G1_N1,0.990183,1.0,0.990183,0.983904,1.0,0.983904,NaN,NaN,NaN,0.774715,0.818693,NaN
1,Lab1_G1_N2,0.994609,1.0,0.994609,0.989325,1.0,0.989325,NaN,NaN,NaN,0.777170,0.850767,NaN
2,Lab1_G1_N3,0.148135,1.0,0.148135,0.136704,1.0,0.136704,NaN,NaN,NaN,0.030118,0.028786,NaN
3,Lab1_G1_P2,0.864980,1.0,0.864980,0.915026,1.0,0.915026,NaN,NaN,NaN,0.500484,0.725153,NaN
4,Lab1_G1_T1,0.993162,1.0,0.993162,0.989900,1.0,0.989900,NaN,NaN,NaN,0.831260,0.870121,NaN


In [43]:
def classify_linear_dependence(max_abs_pearson):
    """
    Качественная интерпретация линейной зависимости по максимальной корреляции Пирсона.
    Пороги эвристические, нужны для EDA.
    """
    if pd.isna(max_abs_pearson):
        return "не определено"
    elif max_abs_pearson >= 0.7:
        return "сильная линейная лаговая связь"
    elif max_abs_pearson >= 0.3:
        return "умеренная линейная лаговая связь"
    else:
        return "слабая линейная лаговая связь"


def classify_possible_nonlinearity(row):
    """
    Эвристика:
    если общая зависимость по NMI заметная,
    но линейная корреляция при этом слабая,
    то связь может быть нелинейной.
    """
    if pd.isna(row["max_nmi"]):
        return "не оценивалось"

    if row["max_nmi"] >= 0.20 and abs(row["pearson_at_max_nmi"]) < 0.30:
        return "возможна нелинейная зависимость"
    elif row["max_nmi"] >= 0.20 and abs(row["pearson_at_max_nmi"]) >= 0.30:
        return "есть зависимость, частично линейная"
    elif row["max_nmi"] < 0.20:
        return "слабая общая зависимость"
    else:
        return "не определено"


linearity_summary["linear_dependence_type"] = linearity_summary["max_abs_pearson"].apply(
    classify_linear_dependence
)

linearity_summary["nonlinear_dependence_type"] = linearity_summary.apply(
    classify_possible_nonlinearity,
    axis=1
)

display(linearity_summary)

,column,max_abs_pearson,tau_max_pearson,pearson_at_best_tau,max_abs_spearman,tau_max_spearman,spearman_at_best_tau,max_nmi,tau_max_nmi,pearson_at_max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi,linear_dependence_type,nonlinear_dependence_type
0,Lab1_G1_N1,0.990183,1.0,0.990183,0.983904,1.0,0.983904,NaN,NaN,NaN,0.774715,0.818693,NaN,сильная линейная лаговая связь,не оценивалось
1,Lab1_G1_N2,0.994609,1.0,0.994609,0.989325,1.0,0.989325,NaN,NaN,NaN,0.777170,0.850767,NaN,сильная линейная лаговая связь,не оценивалось
2,Lab1_G1_N3,0.148135,1.0,0.148135,0.136704,1.0,0.136704,NaN,NaN,NaN,0.030118,0.028786,NaN,слабая линейная лаговая связь,не оценивалось
3,Lab1_G1_P2,0.864980,1.0,0.864980,0.915026,1.0,0.915026,NaN,NaN,NaN,0.500484,0.725153,NaN,сильная линейная лаговая связь,не оценивалось
4,Lab1_G1_T1,0.993162,1.0,0.993162,0.989900,1.0,0.989900,NaN,NaN,NaN,0.831260,0.870121,NaN,сильная линейная лаговая связь,не оценивалось
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,Lab1_TposleNag,0.961746,1.0,0.961746,0.943466,1.0,0.943466,NaN,NaN,NaN,0.943788,0.924434,NaN,сильная линейная лаговая связь,не оценивалось
80,Lab1_Ttg,0.983451,1.0,0.983451,0.978331,1.0,0.978331,NaN,NaN,NaN,0.893006,0.869185,NaN,сильная линейная лаговая связь,не оценивалось
81,Lab1_dPmg,0.776197,1.0,0.776197,0.799281,1.0,0.799281,NaN,NaN,NaN,0.715236,0.750369,NaN,сильная линейная лаговая связь,не оценивалось
82,Lab1_dev,0.548424,1.0,0.548424,0.550815,1.0,0.550815,NaN,NaN,NaN,0.468004,0.479533,NaN,умеренная линейная лаговая связь,не оценивалось


In [44]:
display(linearity_summary["linear_dependence_type"].value_counts().to_frame("count"))

display(linearity_summary["nonlinear_dependence_type"].value_counts().to_frame("count"))

display(
    linearity_summary[
        [
            "max_abs_pearson",
            "max_abs_spearman",
            "max_nmi",
            "mean_abs_pearson",
            "mean_abs_spearman",
            "mean_nmi",
        ]
    ].describe()
)

,count
linear_dependence_type,
сильная линейная лаговая связь,50
слабая линейная лаговая связь,19
умеренная линейная лаговая связь,15


,count
nonlinear_dependence_type,
не оценивалось,84


,max_abs_pearson,max_abs_spearman,max_nmi,mean_abs_pearson,mean_abs_spearman,mean_nmi
count,84.000000,84.000000,0.0,84.000000,84.000000,0.0
mean,0.675867,0.659745,NaN,0.462470,0.479817,NaN
std,0.339527,0.338861,NaN,0.330967,0.341040,NaN
min,0.063260,0.063070,NaN,0.017758,0.019170,NaN
25%,0.387973,0.335088,NaN,0.123201,0.114310,NaN
50%,0.850462,0.820613,NaN,0.447666,0.486961,NaN
75%,0.977479,0.973372,NaN,0.777170,0.819353,NaN
max,0.999673,0.996839,NaN,0.996013,0.979670,NaN


# Задание 3
---
## Вложение временного ряда
На основе выводов из п.2 преобразуйте одномерный временной ряд в многомерное облако точек.
- Равномерное вложение (Uniform Embedding):
  - Определите оптимальные временную задержку и размерность вложения.
- Неравномерное вложение (Non-uniform Embedding):
  - Определите оптимальные различные (неравномерные) шаги задержки (размерность определяется одновременно).

Анализ: Визуализируйте полученные облака точек (в 2D или 3D проекциях, например, через PCA/UMAP). Сравните структуру аттракторов, полученных равномерным и неравномерным методами.
